<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 14: Kisisel Asistan

**YAPAY ZEKA MÜHENDİSLİĞİ** · Modül 14 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta14/hafta14_kisisel_asistan.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta14/hafta14_kisisel_asistan.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 14 - Kişisel Asistan Chatbot Geliştirme

Bu defterde Gemini API kullanarak farklı kişiliklere sahip chatbot'lar oluşturacağız.

## Öğrenme Hedefleri
- `system_instruction` ile kişilik (persona) tanımlama
- Sohbet döngüsü (chat loop) oluşturma
- Bağlam yönetimi ve sohbet geçmişi
- Farklı asistan türleri: Öğretmen, Doktor, Rehber
- Yanıt formatlama
- Bellek ve bağlam penceresi kavramı

In [ ]:
!pip install -q google-generativeai

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `google` | Google Gemini AI API |


In [ ]:
import google.generativeai as genai

API_KEY = "YOUR_API_KEY"  # <-- Kendi anahtarınızı yazın
genai.configure(api_key=API_KEY)

print("API yapılandırıldı!")

## 1. Eğitim Danışmanı Chatbot

İlk chatbot'umuz bir Türk eğitim danışmanı olacak. `system_instruction` parametresi ile modele detaylı bir kişilik tanımlıyoruz.

In [ ]:
# Eğitim danışmanı persona tanımı
egitim_danismani_talimat = """Sen deneyimli bir Türk eğitim danışmanısın. Adın Ayşe Hoca.

Özelliklerin:
- 15 yıllık deneyime sahipsin
- Yapay zeka ve veri bilimi alanında uzmansın
- Öğrencilere kariyer yönlendirmesi yaparsın
- Sabırlı, motive edici ve pozitif bir dilin var
- Türkiye'deki üniversiteler ve sektör hakkında bilgin var

Kuralların:
- Her zaman Türkçe yanıt ver
- Yanıtlarını madde madde ve düzenli ver
- Öğrenciyi cesaretlendir
- Gerçekçi tavsiyeler ver
- Kaynak önerileri ekle
"""

egitim_model = genai.GenerativeModel(
    'gemini-2.5-flash',
    system_instruction=egitim_danismani_talimat
)

print("Eğitim danışmanı chatbot oluşturuldu!")

### Tek soruluk test

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Tek soruluk test
chat = egitim_model.start_chat(history=[])

response = chat.send_message("Merhaba, veri bilimi alanında kariyer yapmak istiyorum. Nereden başlamalıyım?")
print(response.text)

### Takip sorusu - bağlamı koruyor

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Takip sorusu - bağlamı koruyor
response = chat.send_message("Python öğrenmek için hangi kaynakları önerirsin?")
print(response.text)

## 2. Sohbet Döngüsü (Chat Loop)

Gerçek bir chatbot deneyimi için `while True` döngüsü kullanırız. Kullanıcı "çıkış" yazana kadar sohbet devam eder.

> **Not:** Bu hücreyi çalıştırdığınızda interaktif bir sohbet başlar. Çıkmak için `çıkış` yazın.

In [ ]:
def chatbot_calistir(model, baslik="Chatbot"):
    """İnteraktif chatbot döngüsü"""
    print(f"{'='*50}")
    print(f"  {baslik}")
    print(f"  Çıkmak için 'çıkış' yazın")
    print(f"{'='*50}")
    print()
    
    chat = model.start_chat(history=[])
    
    while True:
        kullanici_mesaji = input("Siz: ").strip()
        
        if kullanici_mesaji.lower() in ["çıkış", "cikis", "quit", "exit"]:
            print("\nGörüşmek üzere! İyi çalışmalar! 🎓")
            break
        
        if not kullanici_mesaji:
            print("(Boş mesaj, tekrar deneyin)")
            continue
        
        try:
            response = chat.send_message(kullanici_mesaji)
            print(f"\nAsistan: {response.text}")
            print()
        except Exception as e:
            print(f"\nHata oluştu: {e}")
            print("Tekrar deneyin.\n")
    
    return chat

# Chatbot'u çalıştır
# sohbet = chatbot_calistir(egitim_model, "Eğitim Danışmanı Ayşe Hoca")

## 3. Bağlam Yönetimi ve Sohbet Geçmişi

Gemini'nin sohbet nesnesi (`chat`) önceki mesajları `history` listesinde saklar. Her yeni mesajda tüm geçmiş modele gönderilir.

### Bağlam Penceresi (Context Window)

```
┌─────────────────────────────────────────────┐
│           BAĞLAM PENCERESİ (1M token)       │
│                                             │
│  [Sistem Talimatı]                          │
│  [Mesaj 1: Kullanıcı]                       │
│  [Mesaj 1: Model Yanıtı]                    │
│  [Mesaj 2: Kullanıcı]                       │
│  [Mesaj 2: Model Yanıtı]                    │
│  ...                                        │
│  [Mesaj N: Kullanıcı]  ← Yeni mesaj         │
│                                             │
│  Token sınırı aşılırsa eski mesajlar         │
│  silinmeli veya özetlenmelidir               │
└─────────────────────────────────────────────┘
```

In [ ]:
# Bağlam yönetimi örneği
chat = egitim_model.start_chat(history=[])

# Birkaç mesaj gönderelim
mesajlar = [
    "Merhaba, adım Ali. 20 yaşındayım.",
    "Bilgisayar mühendisliği 2. sınıf öğrencisiyim.",
    "Yapay zeka alanında uzmanlaşmak istiyorum.",
    "Adımı hatırlıyor musun? Hangi bölümde okuyorum?"
]

for mesaj in mesajlar:
    print(f"Kullanıcı: {mesaj}")
    response = chat.send_message(mesaj)
    print(f"Asistan: {response.text[:200]}...")
    print("---")

### Sohbet geçmişini analiz etme

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Sohbet geçmişini analiz etme
print(f"Toplam mesaj sayısı: {len(chat.history)}")
print(f"Token sayısı: {egitim_model.count_tokens(chat.history).total_tokens}")
print()

for i, msg in enumerate(chat.history):
    role = "Kullanıcı" if msg.role == "user" else "Asistan"
    text_preview = msg.parts[0].text[:80]
    print(f"[{i+1}] {role}: {text_preview}...")

## 4. Farklı Asistan Kişilikleri

### 4a. Sağlık Danışmanı

In [ ]:
saglik_talimat = """Sen bilgilendirici bir sağlık danışmanısın. Adın Dr. Elif.

Özelliklerin:
- Genel sağlık bilgisi sağlarsın
- Sağlıklı yaşam tavsiyeleri verirsin
- Beslenme ve egzersiz konularında yol gösterirsin

ÖNEMLİ KURALLAR:
- ASLA tıbbi teşhis koyma
- ASLA ilaç önerme
- Her yanıtın sonuna "Bu bilgiler genel sağlık bilgilendirmesidir. Şikayetleriniz için mutlaka bir doktora başvurun." uyarısını ekle
- Acil durumlarda 112'yi aramalarını söyle
"""

saglik_model = genai.GenerativeModel(
    'gemini-2.5-flash',
    system_instruction=saglik_talimat
)

chat_saglik = saglik_model.start_chat(history=[])
response = chat_saglik.send_message("Bağışıklık sistemimi güçlendirmek için ne yapmalıyım?")
print(response.text)

### 4b. Gezi Rehberi

### Gemini API Kullanımı

Google Gemini modeli ile metin üretimi yapıyoruz. Model, verilen prompt'a göre insan benzeri yanıtlar oluşturur.

In [ ]:
rehber_talimat = """Sen heyecanlı bir Türkiye gezi rehberisin. Adın Mehmet Rehber.

Özelliklerin:
- Türkiye'nin tüm illerini ve turistik yerlerini bilirsin
- Yerel yemekleri, kültürü ve tarihi anlatırsın
- Bütçe dostu öneriler verirsin
- Seyahat ipuçları paylaşırsın
- Coşkulu ve eğlenceli bir dil kullanırsın

Yanıt formatı:
- Her yer için: konum, en iyi zaman, tahmini bütçe, mutlaka yapılması gerekenler
- Yerel yemek önerileri ekle
- Ulaşım bilgisi ver
"""

rehber_model = genai.GenerativeModel(
    'gemini-2.5-flash',
    system_instruction=rehber_talimat
)

chat_rehber = rehber_model.start_chat(history=[])
response = chat_rehber.send_message("Kapadokya'ya gitmek istiyorum, önerilerini alabilir miyim?")
print(response.text)

### 4c. Kodlama Öğretmeni

### Gemini API Kullanımı

Google Gemini modeli ile metin üretimi yapıyoruz. Model, verilen prompt'a göre insan benzeri yanıtlar oluşturur.

In [ ]:
kodlama_talimat = """Sen sabırlı bir Python kodlama öğretmenisin. Adın Kod Ustası Kemal.

Özelliklerin:
- Başlangıç seviyesinden ileri seviyeye kadar Python öğretirsin
- Her konuyu basit örneklerle açıklarsın
- Kodları adım adım, yorum satırlarıyla açıklarsın
- Hata ayıklama (debugging) tekniklerini öğretirsin

Kuralların:
- Kod örnekleri her zaman çalışır durumda olsun
- Her kodun çıktısını göster
- "Kendin Dene" bölümü ekle (küçük alıştırma)
- Türkçe değişken isimleri kullan (ogrenci_notu, toplam_puan gibi)
"""

kodlama_model = genai.GenerativeModel(
    'gemini-2.5-flash',
    system_instruction=kodlama_talimat
)

chat_kodlama = kodlama_model.start_chat(history=[])
response = chat_kodlama.send_message("Python'da listeler (list) konusunu bana öğretir misin?")
print(response.text)

## 5. Yanıtları Güzel Formatlama

Chatbot yanıtlarını daha okunabilir hale getirmek için bir formatlama fonksiyonu yazalım.

In [ ]:
from IPython.display import Markdown, display

def guzel_yanit(response, baslik=None):
    """Gemini yanıtını güzel formatla ve göster."""
    if baslik:
        display(Markdown(f"### {baslik}"))
    display(Markdown(response.text))

def sohbet_gonder(chat, mesaj, baslik=None):
    """Mesaj gönder ve güzel formatla."""
    print(f"Siz: {mesaj}")
    print()
    response = chat.send_message(mesaj)
    guzel_yanit(response, baslik)
    return response

### Güzel formatlı yanıt örneği

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Güzel formatlı yanıt örneği
chat = egitim_model.start_chat(history=[])
sohbet_gonder(chat, "Veri bilimi için öğrenme yol haritası çıkar", baslik="Eğitim Danışmanı Yanıtı")

## 6. Bellek ve Bağlam Penceresi Yönetimi

### Problem: Uzun Sohbetlerde Token Sınırı

Sohbet uzadıkça token sayısı artar. Bağlam penceresi dolduğunda eski mesajları yönetmemiz gerekir.

### Çözüm Stratejileri:
1. **Kayan Pencere (Sliding Window):** Sadece son N mesajı tut
2. **Özetleme:** Eski mesajları özetle
3. **Seçici Hafıza:** Önemli bilgileri ayır

In [ ]:
class AkilliChatbot:
    """Bağlam yönetimli akıllı chatbot."""
    
    def __init__(self, model, max_gecmis=10):
        self.model = model
        self.chat = model.start_chat(history=[])
        self.max_gecmis = max_gecmis  # Maksimum geçmiş mesaj çifti
        self.onemli_bilgiler = []  # Kullanıcı hakkında önemli bilgiler
    
    def mesaj_gonder(self, mesaj):
        """Mesaj gönder ve geçmişi yönet."""
        # Geçmiş çok uzunsa kırp
        if len(self.chat.history) > self.max_gecmis * 2:
            print("[Sistem: Geçmiş kırpılıyor...]")
            # Son N mesaj çiftini tut
            self.chat.history = self.chat.history[-(self.max_gecmis * 2):]
        
        response = self.chat.send_message(mesaj)
        return response
    
    def bilgi_kaydet(self, bilgi):
        """Kullanıcı hakkında önemli bilgi kaydet."""
        self.onemli_bilgiler.append(bilgi)
    
    def durum_raporu(self):
        """Chatbot durumunu göster."""
        token_sayisi = self.model.count_tokens(self.chat.history).total_tokens
        print(f"Mesaj sayısı: {len(self.chat.history)}")
        print(f"Token sayısı: {token_sayisi}")
        print(f"Kaydedilen bilgiler: {len(self.onemli_bilgiler)}")

# Kullanım
akilli_bot = AkilliChatbot(egitim_model, max_gecmis=5)

# Birkaç mesaj gönderelim
response = akilli_bot.mesaj_gonder("Merhaba, adım Zeynep. Fizik öğretmeniyim.")
print(f"Asistan: {response.text[:200]}...")
print()

akilli_bot.bilgi_kaydet("Kullanıcı adı: Zeynep, Meslek: Fizik öğretmeni")
akilli_bot.durum_raporu()

## 7. Çok Kişilikli Asistan Sistemi

Farklı konularda farklı uzman chatbot'lara yönlendiren bir sistem oluşturalım.

In [ ]:
class AsistanYonetici:
    """Birden fazla uzman asistanı yöneten sistem."""
    
    def __init__(self):
        self.asistanlar = {}
        self.aktif_asistan = None
    
    def asistan_ekle(self, isim, talimat, aciklama):
        """Yeni bir uzman asistan ekle."""
        model = genai.GenerativeModel(
            'gemini-2.5-flash',
            system_instruction=talimat
        )
        self.asistanlar[isim] = {
            'model': model,
            'chat': model.start_chat(history=[]),
            'aciklama': aciklama
        }
    
    def asistanlari_listele(self):
        """Mevcut asistanları listele."""
        print("Mevcut Asistanlar:")
        print("-" * 40)
        for isim, bilgi in self.asistanlar.items():
            print(f"  [{isim}] - {bilgi['aciklama']}")
    
    def sor(self, asistan_ismi, mesaj):
        """Belirli bir asistana soru sor."""
        if asistan_ismi not in self.asistanlar:
            print(f"'{asistan_ismi}' adında bir asistan bulunamadı!")
            return None
        
        asistan = self.asistanlar[asistan_ismi]
        response = asistan['chat'].send_message(mesaj)
        return response

# Sistem oluştur ve asistanları ekle
sistem = AsistanYonetici()

sistem.asistan_ekle(
    "egitim",
    egitim_danismani_talimat,
    "Eğitim ve kariyer danışmanı"
)

sistem.asistan_ekle(
    "saglik",
    saglik_talimat,
    "Sağlık bilgilendirme danışmanı"
)

sistem.asistan_ekle(
    "gezi",
    rehber_talimat,
    "Türkiye gezi rehberi"
)

sistem.asistanlari_listele()

### Farklı asistanlara soru sorma

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Farklı asistanlara soru sorma
print("=== EĞİTİM DANIŞMANI ===")
r = sistem.sor("egitim", "Veri bilimi için hangi sertifikalar önemli?")
print(r.text[:300])
print()

print("=== GEZİ REHBERİ ===")
r = sistem.sor("gezi", "Antalya'da 3 günlük bir plan öner")
print(r.text[:300])

## Özet

Bu defterde öğrendiklerimiz:

| Konu | Açıklama |
|------|----------|
| **Persona Tanımlama** | `system_instruction` ile detaylı kişilik oluşturma |
| **Sohbet Döngüsü** | `while True` ile interaktif chatbot |
| **Bağlam Yönetimi** | Sohbet geçmişini takip etme ve yönetme |
| **Farklı Asistanlar** | Eğitim, sağlık, gezi gibi farklı uzman botlar |
| **Formatlama** | `IPython.display.Markdown` ile güzel gösterim |
| **Bellek Yönetimi** | Kayan pencere ve özetleme stratejileri |
| **Çoklu Asistan** | Birden fazla uzmanı yöneten sistem |

### Alıştırma
Kendi chatbot'unuzu oluşturun! Bir konu seçin (spor koçu, yemek tarifi uzmanı, film eleştirmeni vb.) ve detaylı bir `system_instruction` yazın.

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://akademikyz.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>